### Order Book Requirements
- Maintain Bid side & Ask side (price, size)
- Tick size, Trade history
- Expected Invariants:
  - Best bid < Best Ask
  - No negative sizes
  - Prices aligned to tick

### Event processing
- Limit Order with
  * input: side=buy/sell, price, size
  * logic: adds liquidity, also if limit order crosses spread-> convert to market order
- Market Order:
  * input: side, size
  * logic: consumes relevant side liquidity, may sweep multiple price levels, moves price if best level depleted
  * Market order size > total opppositve liquidity. Todo: think!
- Cancellation
  * inpute: side, price, size
  * logic: reduces liquidity, if level zero in size-> remove price level

### Trade history recording 
Important for lambda, Trade imbalance, etc
input: side, price, size


In [6]:
"""
Assumption: no queueing right now for any price level!
Given the event processing inputs, we can take streams of order 
tuples (order_type(limit/cancel/market), side(buy/sell), price, size)

L: Limit Order
M: Market Order
X: Cancel Order

B: Buy
S: Sell
"""

events = [
    ('L', 'B', 100.05, 40),
    ('L', 'S', 100.08, 100),
    ('M', 'B', None, 50),
    ('X', 'B', 100.05, 10)
]

In [40]:
"""
raw events is not good to be stored like that. challenges: not efficient to query best bid/ask, 
size of a price also not efficient to query for adding orders/cancelling it.

Best bid: query should be fast, raw input events may take O(n) to find! We can sort it to get O(1).
Modifying limit order size: for this sorting alone don't help, because cancel order may be for not best price. So, for storing
size we can use Dict.

For adding new price level: O(N), we can limit N=20 price levels
"""
# Current state
bid_levels = {
    100.05: 40,
    100.03: 20,
    100.02: 1000
}

ask_levels = {
    100.06: 60,
    100.07: 5600,
    100.08: 100
}

bids = list(bid_levels.keys())
asks = list(ask_levels.keys())

bids.sort()
asks.sort()
print(bids,'\n', asks)

[100.02, 100.03, 100.05] 
 [100.06, 100.07, 100.08]


In [36]:
# process_events(): take events and add to data structure for bids and asks
def process_event(event):
    # event tuple (Order Type, Buy/Sell, Price, Size)
    # if type is Market order expects price = None
    order_type, side, price, size = event
    if order_type=='L' and ((side=='B' and price>=asks[0]) or (side=='S' and price<=bids[-1])):
        if side=='B':
            while size and asks:
                best_ask=asks[0]
                if best_ask>price:
                    break
                size_available = ask_levels[best_ask]
                if size>=size_available:
                    ask_levels.pop(best_ask)
                    asks.remove(best_ask)
                    size-=size_available
                else:
                    ask_levels[best_ask]-=size
                    break
        else:
            while size and bids:
                best_bid=bids[-1]
                if best_bid<price:
                    break
                size_available = bid_levels[best_bid]
                if size>=size_available:
                    bid_levels.pop(best_bid)
                    bids.remove(best_bid)
                    size-=size_available
                else:
                    bid_levels[best_bid]-=size
                    break
    elif order_type == 'L':
        if side == 'B':
            # Limit buy order
            if price in bid_levels:
                bid_levels[price]+=size
            else:
                bids.append(price)
                bids.sort()
                bid_levels[price]=size
        else: # side == 'S'
            # limit sell order
            if price in ask_levels:
                ask_levels[price]+=size
            else:
                asks.append(price)
                asks.sort()
                ask_levels[price]=size
    elif order_type == 'X':
        if side == 'B':
            # Cancel buy order
            if price in bid_levels:
                bid_levels[price]=max(bid_levels[price]-size,0) #size>=0 constraint in each level
                if bid_levels[price] == 0:
                    bid_levels.pop(price)
                    bids.remove(price)
        else: # side == 'S'
            # Cancel sell order
            if price in ask_levels:
                ask_levels[price]=max(ask_levels[price]-size,0)
                if ask_levels[price] == 0:
                    ask_levels.pop(price)
                    asks.remove(price)
    else: # order type 'M' Market Order
        if side=='B':
            while size and asks:
                best_ask=asks[0]
                size_available = ask_levels[best_ask]
                if size>=size_available:
                    ask_levels.pop(best_ask)
                    asks.remove(best_ask)
                    size-=size_available
                else:
                    ask_levels[best_ask]-=size
                    break
        else:
            while size and bids:
                best_bid=bids[-1]
                size_available = bid_levels[best_bid]
                if size>=size_available:
                    bid_levels.pop(best_bid)
                    bids.remove(best_bid)
                    size-=size_available
                else:
                    bid_levels[best_bid]-=size
                    break

In [41]:
def print_state():
    print("Bids", bids, bid_levels)
    print("Asks", asks, ask_levels)
    print("-----")
print_state()

Bids [100.02, 100.03, 100.05] {100.05: 40, 100.03: 20, 100.02: 1000}
Asks [100.06, 100.07, 100.08] {100.06: 60, 100.07: 5600, 100.08: 100}
-----


In [42]:
for event in events:
    print(event)
    process_event(event)
    print_state()

('L', 'B', 100.05, 40)
Bids [100.02, 100.03, 100.05] {100.05: 80, 100.03: 20, 100.02: 1000}
Asks [100.06, 100.07, 100.08] {100.06: 60, 100.07: 5600, 100.08: 100}
-----
('L', 'S', 100.08, 100)
Bids [100.02, 100.03, 100.05] {100.05: 80, 100.03: 20, 100.02: 1000}
Asks [100.06, 100.07, 100.08] {100.06: 60, 100.07: 5600, 100.08: 200}
-----
('M', 'B', None, 50)
Bids [100.02, 100.03, 100.05] {100.05: 80, 100.03: 20, 100.02: 1000}
Asks [100.06, 100.07, 100.08] {100.06: 10, 100.07: 5600, 100.08: 200}
-----
('X', 'B', 100.05, 10)
Bids [100.02, 100.03, 100.05] {100.05: 70, 100.03: 20, 100.02: 1000}
Asks [100.06, 100.07, 100.08] {100.06: 10, 100.07: 5600, 100.08: 200}
-----


### The Standard Industry Pattern:

1. **Price Map (The Ladder):** A **Balanced Binary Search Tree (BST)** or a **Skiplist** (in C++ often `std::map`).
* *Why:* Keeps prices sorted automatically. Insertion and deletion are $O(\log M)$, where $M$ is the number of price levels.


2. **Price Level (The Queue):** A **Doubly Linked List** of Order objects at each price level.
* *Why:* Real exchange rules are **FIFO** (First-In, First-Out). Adding to the end or removing from the front is $O(1)$.


3. _Not implemented here_ **Order Map:** A **Hash Map** (`dict`) of `OrderID -> OrderObject`
* *Why:* When a "Cancel" (`X`) comes in, you usually get an `OrderID`. You need to find that order instantly ($O(1)$) without scanning the whole book.

In [3]:
from sortedcontainers import SortedDict

asks = SortedDict() # ascending order
bids = SortedDict() # we can store negative price to get it sorted

In [7]:
asks[100.0] = 20 # Price -> Volume
asks[100.1] = 30
asks

SortedDict({100.0: 20, 100.1: 30})

In [9]:
best_price asks.iloc[0]

100.0

In [18]:
bids[-99.8]=40
bids[-99.6]=100
bids[-99.7]=40
bids

SortedDict({-99.8: 40, -99.7: 40, -99.6: 100})

In [21]:
best_bid = abs(bids.iloc[0])
best_bid

99.8

In [24]:
# order depth
bids.iloc[0:20]

[-99.8, -99.7, -99.6]

In [25]:
asks.iloc[0:20]

[100.0, 100.1]

In [26]:
empty_dict = SortedDict()
if empty_dict:
    print("Evaluate to true")

In [34]:
# Matching orders - Execute orders which can be immediately trader i.e. market orders and crossing spread limit orders (within price limits)
def match(side, size, price=None):
    # Match against opposing book
    opposing_book = asks if side == 'B' else bids
    while size > 0 and opposing_book:
       best_key = opposing_book.iloc[0]
       best_price = abs(best_key) 
       
       # Price protection logic:
       if price is not None:
           if side == 'B' and best_price > price: break #If buy, don't pay more incase of limit orders
           if side == 'S' and best_price < price: break
               
       available = opposing_book[best_key]
       if size >= available:
           size -= available
           opposing_book.pop(best_key)
       else:
           opposing_book[best_key] -= size
           size = 0
    return size
    

In [36]:
print('bids=',bids)
print('asks=',asks)

bids= SortedDict({-99.8: 40, -99.7: 40, -99.6: 100})
asks= SortedDict({100.0: 20, 100.1: 30})


In [37]:
match('B', 20) # Market order - executes immediately
print(bids, asks)

SortedDict({-99.8: 40, -99.7: 40, -99.6: 100}) SortedDict({100.1: 30})


In [38]:
asks[100.2]=100

In [39]:
print(bids, asks)

SortedDict({-99.8: 40, -99.7: 40, -99.6: 100}) SortedDict({100.1: 30, 100.2: 100})


In [40]:
match('B', 20, 99.85) # buy limit order without crossing spread
print(bids, asks) # Nothing happens bcz we have not written to add new price level for this

SortedDict({-99.8: 40, -99.7: 40, -99.6: 100}) SortedDict({100.1: 30, 100.2: 100})


In [41]:
_ = match('B', 60, 100.1) # buy limit order crossing spread but high vol 
print('leftover size', _)
print(bids, asks)

leftover size 30
SortedDict({-99.8: 40, -99.7: 40, -99.6: 100}) SortedDict({100.2: 100})


In [47]:
import sys
import os

target_dir = os.path.abspath('../src/')
sys.path.insert(0, target_dir) 

In [48]:
from orderbook import OrderBook

In [51]:
ob = OrderBook()
ob

In [52]:
ob.bids

SortedDict({})

In [70]:
ob.bids= SortedDict({-99.8: 40, -99.7: 40, -99.6: 100})
ob.asks= SortedDict({100.0: 20, 100.1: 30})

In [71]:
events = [
    ('L', 'B', 100.05, 40),
    ('L', 'S', 100.08, 100),
    ('M', 'B', None, 50),
    ('X', 'B', 100.05, 10)
]

for e in events:
    ob.process_event(e)
    print(ob.bids, ob.asks, 'spread=', round(ob.asks.iloc[0]+ob.bids.iloc[0],2))

SortedDict({-100.05: 20, -99.8: 40, -99.7: 40, -99.6: 100}) SortedDict({100.1: 30}) spread= 0.05
SortedDict({-100.05: 20, -99.8: 40, -99.7: 40, -99.6: 100}) SortedDict({100.08: 100, 100.1: 30}) spread= 0.03
SortedDict({-100.05: 20, -99.8: 40, -99.7: 40, -99.6: 100}) SortedDict({100.08: 50, 100.1: 30}) spread= 0.03
SortedDict({-100.05: 10, -99.8: 40, -99.7: 40, -99.6: 100}) SortedDict({100.08: 50, 100.1: 30}) spread= 0.03


/var/folders/10/vzvbs3px76sfj57d88__6yjc0000gn/T/ipykernel_753/3447960309.py:10: DeprecationWarning: sorted_dict.iloc is deprecated. Use SortedDict.keys() instead.
  print(ob.bids, ob.asks, 'spread=', round(ob.asks.iloc[0]+ob.bids.iloc[0],2))


In [65]:
# .peekitem(0)[0] instead of iloc[0]